# 🚀 AdamOps Framework — Full Integration Test

Comprehensive testing of **all** core modules using the New York Restaurants dataset.

## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import warnings
import os, sys, tempfile, json
warnings.filterwarnings('ignore')

import adamops
from adamops.data.loaders import load_csv
from adamops.data.preprocessors import preprocess
from adamops.data.splitters import split_train_test
from adamops.models.modelops import train, train_regression, train_classification
from adamops.models.ensembles import create_voting_ensemble
from adamops.models.automl import run as automl_run
from adamops.evaluation.metrics import evaluate_model, compare_results

print(f'✅ All imports successful. AdamOps v{adamops.__version__}')

## 2. Data Loading & Preprocessing

In [ ]:
# Load the New York Restaurants dataset
data_path = '../test-data/newyork-rest.csv'
df = load_csv(data_path)
print(f'Loaded: {df.shape[0]} rows x {df.shape[1]} columns')

# Drop non-numeric and irrelevant columns for price prediction
df_clean = df.drop(columns=['Case', 'Restaurant'])
df_preprocessed = preprocess(df_clean, missing_strategy="mean", outlier_method="iqr", remove_duplicates=True)
print(f'Preprocessed: {df_preprocessed.shape}')
df_preprocessed.head()

## 3. Feature Engineering & Train/Test Split

In [ ]:
# Split the data into features (X) and target (y)
X = df_preprocessed.drop(columns=['Price'])
y = df_preprocessed['Price']

X_train, X_test, y_train, y_test = split_train_test(X, y, test_size=0.2, random_state=42)
print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')
print(f'y_train: {y_train.shape}, y_test: {y_test.shape}')

## 4. Base Model Training

In [ ]:
# Train Random Forest and Ridge regression models
rf_model = train_regression(X_train, y_train, algorithm='random_forest')
ridge_model = train_regression(X_train, y_train, algorithm='ridge')

print(f'✅ Random Forest fitted: {type(rf_model).__name__}')
print(f'✅ Ridge fitted: {type(ridge_model).__name__}')

## 5. Ensemble Modeling

In [ ]:
# Create and fit a Voting Ensemble
ensemble_model = create_voting_ensemble(algorithms=['random_forest', 'ridge'], task='regression')
ensemble_model.fit(X_train, y_train)

ensemble_preds = ensemble_model.predict(X_test)
print(f'✅ Voting Ensemble fitted. Predictions shape: {ensemble_preds.shape}')

## 6. AutoML (Bayesian Optimization)

In [ ]:
# Run AutoML with Optuna backend
automl_result = automl_run(
    X_train, y_train, 
    task='regression', 
    backend='local',
    algorithms=['random_forest', 'ridge'],
    n_trials=10, 
    cv=3
)
best_automl_model = automl_result.best_model

print(f'✅ AutoML Best Score (Negative MSE): {automl_result.best_score:.4f}')
print(f'✅ AutoML Best Params: {automl_result.best_params}')
print(f'✅ Leaderboard:\n{automl_result.leaderboard}')

## 7. Evaluation & Comparison

In [ ]:
# Evaluate all models
rf_metrics = evaluate_model(rf_model, X_test, y_test, task='regression')
ridge_metrics = evaluate_model(ridge_model, X_test, y_test, task='regression')
ensemble_metrics = evaluate_model(ensemble_model, X_test, y_test, task='regression')
automl_metrics = evaluate_model(best_automl_model, X_test, y_test, task='regression')

comparison_df = compare_results(
    [rf_metrics, ridge_metrics, ensemble_metrics, automl_metrics],
    names=['Random Forest', 'Ridge', 'Voting Ensemble', 'AutoML Best']
)
print('✅ Model Comparison:')
comparison_df

## 8. Hardware Abstraction Layer

Tests DeviceManager: CUDA/MPS/CPU detection, VRAM checks, Colab detection, checkpoint routing.

In [ ]:
from adamops.hardware import DeviceManager

# 1. Device Detection
device = DeviceManager.get_optimal_device()
print(f'Optimal Device: {device}')

# 2. GPU Count
num_gpus = DeviceManager.get_num_gpus()
print(f'Available GPUs: {num_gpus}')

# 3. Colab Environment Detection
is_colab = DeviceManager.is_colab_environment()
print(f'Running in Colab: {is_colab}')

# 4. VRAM Pre-flight Check (safe for CPU, always returns True)
vram_ok = DeviceManager.vram_preflight_check(required_mb=1024.0)
print(f'VRAM Pre-flight (1GB): {vram_ok}')

# 5. Checkpoint Directory Routing
ckpt_dir = DeviceManager.get_checkpoint_dir('test_project')
print(f'Checkpoint Dir: {ckpt_dir}')

# 6. Model VRAM Estimation (sklearn model = 0 MB, uses RAM not GPU)
est_mb = DeviceManager.estimate_model_vram(rf_model)
print(f'Estimated VRAM for RF model: {est_mb:.2f} MB (0 expected for sklearn)')

print('\n✅ Hardware Abstraction Layer: ALL PASSED')

## 9. Distributed Training Manager

Tests DistributedManager: auto-strategy, parallel sklearn, DDP setup.

In [ ]:
from adamops.distributed import DistributedManager

# 1. Auto-Strategy Selection
strategy_pytorch = DistributedManager.auto_strategy(backend='pytorch')
strategy_sklearn = DistributedManager.auto_strategy(backend='sklearn')
print(f'Auto Strategy (PyTorch): {strategy_pytorch}')
print(f'Auto Strategy (sklearn): {strategy_sklearn}')

# 2. Parallel sklearn Training
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

def train_ridge():
    m = Ridge(alpha=1.0); m.fit(X_train, y_train); return m

def train_rf():
    m = RandomForestRegressor(n_estimators=10, random_state=42); m.fit(X_train, y_train); return m

parallel_results = DistributedManager.parallel_sklearn_train(
    [train_ridge, train_rf], n_jobs=2
)
print(f'Parallel Training Results: {len(parallel_results)} models trained')
for i, model in enumerate(parallel_results):
    print(f'  Model {i+1}: {type(model).__name__}, score={model.score(X_test, y_test):.4f}')

print('\n✅ Distributed Training Manager: ALL PASSED')

## 10. Unified Trainer Class

Tests Trainer: sklearn backend fit/predict/evaluate/save lifecycle.

In [ ]:
from adamops.trainer import Trainer

# 1. Initialize Trainer with sklearn backend
trainer = Trainer('random_forest', task='regression', strategy='auto')
print(f'Backend: {trainer.backend}')
print(f'Strategy: {trainer.strategy}')
print(f'Device: {trainer.device}')
print(f'Is Colab: {trainer.is_colab}')

# 2. Fit
trainer.fit(X_train, y_train)
print(f'\nModel Fitted: {type(trainer.trained_model).__name__}')

# 3. Predict
preds = trainer.predict(X_test)
print(f'Predictions Shape: {preds.shape}')
print(f'Sample Predictions: {preds[:5]}')

# 4. Evaluate
metrics = trainer.evaluate(X_test, y_test)
print(f'\nEvaluation Metrics: {metrics}')

# 5. Save (checkpoint routing)
save_path = trainer.save(run_name='test_run')
print(f'\nCheckpoint saved to: {save_path}')
assert os.path.exists(save_path), 'Checkpoint file not found!'

print('\n✅ Unified Trainer Class: ALL PASSED')

## 11. Colab Bridge

Tests ColabBridge initialization, message protocol, and setup snippet (no live Colab required).

In [ ]:
from adamops.colab.bridge import ColabBridge, ColabResult, _make_execute_msg
from adamops.colab.setup_snippet import get_setup_snippet, COLAB_SETUP_SNIPPET

# 1. ColabResult dataclass
result = ColabResult(success=True, elapsed=1.23, stdout='hello from GPU!')
print(f'ColabResult: {result}')
assert result.success is True
assert result.returncode == 0

error_result = ColabResult(success=False, error_name='ValueError', error_value='bad input')
print(f'Error Result: {error_result}')
assert 'bad input' in repr(error_result)

# 2. Kernel Message Protocol
msg = _make_execute_msg("print('test')", 'test-session')
assert msg['header']['msg_type'] == 'execute_request'
assert msg['content']['code'] == "print('test')"
assert msg['header']['username'] == 'adamops'
assert msg['channel'] == 'shell'
print(f'Execute Message: msg_type={msg["header"]["msg_type"]}, channel={msg["channel"]}')

# 3. ColabBridge Initialization
bridge = ColabBridge('http://localhost:8888', token='test-token-123')
assert bridge._base_url == 'http://localhost:8888'
assert bridge._token == 'test-token-123'
assert bridge._headers == {'Authorization': 'token test-token-123'}
print(f'Bridge URL: {bridge._base_url}')

# 4. WebSocket URL Construction
bridge._kernel_id = 'test-kernel'
ws_url = bridge._ws_url()
assert ws_url == 'ws://localhost:8888/api/kernels/test-kernel/channels?token=test-token-123'
print(f'WS URL: {ws_url}')

# HTTPS to WSS conversion
bridge_ssl = ColabBridge('https://colab.example.com', token='tok')
bridge_ssl._kernel_id = 'kern'
assert bridge_ssl._ws_url().startswith('wss://')

# 5. Setup Snippet
snippet = get_setup_snippet()
assert 'jupyter_kernel_gateway' in snippet
assert 'ColabBridge' in snippet
assert len(COLAB_SETUP_SNIPPET) > 100
print(f'Setup Snippet Length: {len(snippet)} chars')

# 6. Error handling for missing files
try:
    bridge.run_script('/nonexistent/path.py')
    assert False, 'Should have raised FileNotFoundError'
except FileNotFoundError:
    print('FileNotFoundError correctly raised for missing script')

try:
    bridge.run_notebook('/nonexistent/notebook.ipynb')
    assert False, 'Should have raised FileNotFoundError'
except FileNotFoundError:
    print('FileNotFoundError correctly raised for missing notebook')

print('\n✅ Colab Bridge: ALL PASSED')

## 12. Backend Engine (StateDB + REST API)

Tests StateDB CRUD operations and FastAPI endpoints.

In [ ]:
from adamops.backend.database import StateDB

# Create an in-memory database
db = StateDB(':memory:')

# 1. Experiments CRUD
exp = db.create_experiment('NY Restaurant Price Prediction', 'Testing ADAMOPS framework')
print(f'Created Experiment: id={exp["id"]}, name={exp["name"]}')
assert exp['name'] == 'NY Restaurant Price Prediction'

fetched = db.get_experiment(exp['id'])
assert fetched is not None
assert fetched['description'] == 'Testing ADAMOPS framework'

all_exps = db.list_experiments()
assert len(all_exps) == 1

# 2. Runs CRUD
run = db.create_run(exp['id'], algorithm='random_forest', task='regression')
print(f'Created Run: id={run["id"]}, algorithm={run["algorithm"]}, status={run["status"]}')
assert run['status'] == 'pending'

db.update_run_status(run['id'], 'running')
assert db.get_run(run['id'])['status'] == 'running'

db.update_run_metrics(run['id'], {'mse': 25.3, 'r2': 0.87, 'mae': 3.1})
run_updated = db.get_run(run['id'])
assert run_updated['metrics']['r2'] == 0.87
print(f'Updated Metrics: {run_updated["metrics"]}')

db.update_run_status(run['id'], 'completed')
completed = db.get_run(run['id'])
assert completed['status'] == 'completed'
assert completed['finished_at'] is not None

# 3. Events
db.log_event(run['id'], 'log', {'message': 'Training started'})
db.log_event(run['id'], 'metrics', {'epoch': 1, 'loss': 0.5})
db.log_event(run['id'], 'metrics', {'epoch': 2, 'loss': 0.3})
events = db.get_events(run['id'])
assert len(events) == 3
print(f'Events logged: {len(events)}')

# 4. Model Registry
model_reg = db.register_model('rf-price-predictor', 'v1', run_id=run['id'], metadata={'accuracy': 0.87})
print(f'Registered Model: {model_reg["name"]} {model_reg["version"]}')
models = db.list_models()
assert len(models) == 1
assert models[0]['metadata']['accuracy'] == 0.87

# 5. Cascade Delete
db.delete_experiment(exp['id'])
assert db.get_experiment(exp['id']) is None
assert db.get_run(run['id']) is None
assert db.get_events(run['id']) == []
print('Cascade delete verified')

print('\n✅ Backend Engine (StateDB): ALL PASSED')

In [ ]:
# FastAPI REST Endpoints Test
import time
from adamops.backend.app import create_app
from adamops.backend.database import StateDB
from fastapi.testclient import TestClient
from sklearn.datasets import make_classification

test_db = StateDB(':memory:')
app = create_app(test_db)
client = TestClient(app)

# Health check
r = client.get('/')
assert r.status_code == 200
assert r.json()['status'] == 'healthy'
print(f'Health: {r.json()}')

# Create experiment via REST
r = client.post('/api/experiments', json={'name': 'REST Test', 'description': 'via API'})
assert r.status_code == 201
exp_id = r.json()['experiment']['id']
print(f'Created Experiment via REST: id={exp_id}')

# Submit run with inline data (triggers background execution)
X_demo, y_demo = make_classification(n_samples=50, n_features=5, random_state=42)

r = client.post('/api/runs', json={
    'experiment_id': exp_id,
    'algorithm': 'random_forest',
    'task': 'classification',
    'features': X_demo.tolist(),
    'target': y_demo.tolist(),
})
assert r.status_code == 201
run_id = r.json()['run']['id']
print(f'Submitted Run: id={run_id}')

# Wait for background execution
time.sleep(3)

r = client.get(f'/api/runs/{run_id}')
run_result = r.json()['run']
print(f'Run Status: {run_result["status"]}')
print(f'Run Metrics: {run_result.get("metrics", {})}')
assert run_result['status'] == 'completed'
assert 'accuracy' in run_result['metrics']

# Check events
r = client.get(f'/api/runs/{run_id}/events')
events = r.json()['events']
print(f'Events: {len(events)} logged')
assert len(events) > 0

# Register model via REST
r = client.post('/api/models', json={
    'name': 'test-model', 'version': 'v1',
    'metadata': {'accuracy': run_result['metrics'].get('accuracy', 0)}
})
assert r.status_code == 201
print(f'Registered Model: {r.json()["model"]["name"]}')

print('\n✅ Backend Engine (REST API): ALL PASSED')

## 13. Studio Pipeline Compiler

Tests compile_pipeline(): DAG to Python code generation and end-to-end execution.

In [ ]:
import ast
from adamops.studio.compiler import compile_pipeline
from sklearn.datasets import make_classification

# 1. Empty pipeline error handling
try:
    compile_pipeline({'nodes': [], 'connections': []})
    assert False, 'Should have raised error'
except Exception as e:
    print(f'Empty pipeline error: {e}')

# 2. Compile a full Load -> Split -> Train -> Evaluate pipeline

# Create a temp dataset
X_comp, y_comp = make_classification(n_samples=50, n_features=4, random_state=42)
df_comp = pd.DataFrame(X_comp, columns=[f'f_{i}' for i in range(4)])
df_comp['target'] = y_comp

with tempfile.NamedTemporaryFile(suffix='.csv', mode='w', delete=False) as f:
    df_comp.to_csv(f.name, index=False)
    temp_csv = f.name

pipeline_data = {
    'nodes': [
        {'id': 'load', 'type': 'load_csv', 'params': {'filepath': temp_csv}},
        {'id': 'split', 'type': 'train_test_split', 'params': {'target': 'target', 'test_size': 0.2}},
        {'id': 'model', 'type': 'train_classification', 'params': {'algorithm': 'random_forest'}},
        {'id': 'eval', 'type': 'evaluate', 'params': {}},
    ],
    'connections': [
        {'from_node': 'load', 'from_port': 'dataframe', 'to_node': 'split', 'to_port': 'dataframe'},
        {'from_node': 'split', 'from_port': 'X_train', 'to_node': 'model', 'to_port': 'X_train'},
        {'from_node': 'split', 'from_port': 'y_train', 'to_node': 'model', 'to_port': 'y_train'},
        {'from_node': 'split', 'from_port': 'X_test', 'to_node': 'eval', 'to_port': 'X_test'},
        {'from_node': 'split', 'from_port': 'y_test', 'to_node': 'eval', 'to_port': 'y_test'},
        {'from_node': 'model', 'from_port': 'model', 'to_node': 'eval', 'to_port': 'model'},
    ]
}

code = compile_pipeline(pipeline_data)
print(f'Compiled code length: {len(code)} chars')

# 3. Verify it is valid Python AST
parsed = ast.parse(code)
assert parsed is not None
print('AST validation: Valid Python')

# 4. Execute the compiled code
namespace = {}
exec(code, namespace)
results = namespace['main']()

assert 'eval_metrics' in results
assert 'accuracy' in results['eval_metrics']
print(f'Pipeline Metrics: {results["eval_metrics"]}')
print(f'Pipeline Model: {type(results["model_model"]).__name__}')

# Cleanup
os.unlink(temp_csv)

print('\n✅ Studio Pipeline Compiler: ALL PASSED')

## Summary

In [ ]:
summary = [
    ('Data Loading & Preprocessing', 'PASSED'),
    ('Feature Engineering & Splitting', 'PASSED'),
    ('Base Model Training (RF + Ridge)', 'PASSED'),
    ('Ensemble Modeling (Voting)', 'PASSED'),
    ('AutoML (Optuna Bayesian)', 'PASSED'),
    ('Evaluation & Comparison', 'PASSED'),
    ('Hardware Abstraction Layer', 'PASSED'),
    ('Distributed Training Manager', 'PASSED'),
    ('Unified Trainer Class', 'PASSED'),
    ('Colab Bridge', 'PASSED'),
    ('Backend Engine (StateDB)', 'PASSED'),
    ('Backend Engine (REST API)', 'PASSED'),
    ('Studio Pipeline Compiler', 'PASSED'),
]

summary_df = pd.DataFrame(summary, columns=['Module', 'Status'])
summary_df.index = summary_df.index + 1
print('ADAMOPS FULL INTEGRATION TEST RESULTS')
print('=' * 50)
print(summary_df.to_string())
print(f'\n{len(summary)}/{len(summary)} modules passed.')
print(f'AdamOps v{adamops.__version__} - All systems operational.')